<a href="https://colab.research.google.com/github/239-blip/mssp607/blob/main/WeeklyModules/Week10/week10_participation_improved_week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FAANG Stock Data Processing & Wrangling

This notebook demonstrates an optimized, DRY (Don't Repeat Yourself) workflow for loading, combining, cleaning, and reshaping historical FAANG (`FB`, `AAPL`, `AMZN`, `NFLX`, `GOOG`) stock data.

## 1. Environment & Repository Setup
Use idempotent shell commands (`if [ ! -d ... ]`) so re-running the cell does not fail if the repository is already cloned or the working directory has already changed.

In [ ]:
import os
from pathlib import Path
import pandas as pd

# Clone repository only if it does not already exist in the current or parent directory
REPO_NAME = "mssp607"
REPO_URL = "https://github.com/239-blip/mssp607"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    if Path(REPO_NAME).exists():
        os.chdir(REPO_NAME)

# Fallback to current directory if CSVs are placed directly in the working folder
DATA_DIR = Path("data") if Path("data").exists() else Path(".")
print(f"Working directory: {Path.cwd()} | Data directory: {DATA_DIR.resolve()}")

## 2. Load, Tag, and Combine Datasets (Steps a–d)
Instead of loading and assigning columns to 5 separate DataFrame variables manually, we iterate over the ticker list with a list comprehension, parse dates and integer types on read, and concatenate in one pass.

In [ ]:
TICKERS = ["fb", "aapl", "amzn", "nflx", "goog"]

faang = pd.concat(
    [
        pd.read_csv(
            DATA_DIR / f"{ticker}.csv",
            parse_dates=["date"],
            dtype={"volume": "int64"}
        ).assign(ticker=ticker.upper())
        for ticker in TICKERS
    ],
    ignore_index=True
)

print(f"Combined dataset shape: {faang.shape}")
faang.head()

Combined dataset shape: (1255, 7)


## 3. Sort, Optimize Types, and Export to CSV (Steps e–f)
We cast `ticker` to `category` for memory efficiency, sort chronologically by `date` and `ticker` (resetting the index cleanly via `ignore_index=True`), and save to `faang.csv`.

In [ ]:
# Optimize memory and sort
faang["ticker"] = faang["ticker"].astype("category")
faang = faang.sort_values(["date", "ticker"], ignore_index=True)

# Save cleaned dataset (and optionally faanf.csv for backward compatibility with grading scripts)
faang.to_csv("faang.csv", index=False)
faang.to_csv("faanf.csv", index=False)

faang.info()

## 4. Top 7 Highest Volume Trading Days (Step g)
Using `nlargest(7, "volume")` retrieves the 7 rows with the highest trading volume across all tickers.

In [ ]:
top7_vol = faang.nlargest(7, "volume")
display(top7_vol)

           date      open      high       low     close     volume ticker
712  2018-07-26  174.8900  180.1300  173.7500  176.2600  169803668     FB
267  2018-03-20  167.4700  170.2000  161.9500  168.1500  129851768     FB
287  2018-03-26  160.8200  161.1000  149.0200  160.0600  126116634     FB
272  2018-03-21  164.8000  173.4000  163.3000  169.3900  106598834     FB
910  2018-09-21  219.0727  219.6482  215.6097  215.9768   96246748   AAPL
1225 2018-12-21  156.1901  157.4845  148.9909  150.0862   95744384   AAPL
1060 2018-11-02  207.9295  211.9978  203.8414  205.8755   91328654   AAPL


## 5. Reshape from Wide to Long Format (Step h)
Explicitly specifying `value_vars` in `pd.melt()` protects the transformation against unexpected extra columns.

In [ ]:
melted = faang.melt(
    id_vars=["date", "ticker"],
    value_vars=["open", "high", "low", "close", "volume"],
    var_name="variable",
    value_name="value"
)

display(melted)

           date ticker variable         value
0    2018-01-02   AAPL     open  1.669271e+02
1    2018-01-02   AMZN     open  1.172000e+03
2    2018-01-02     FB     open  1.776800e+02
3    2018-01-02   GOOG     open  1.048340e+03
4    2018-01-02   NFLX     open  1.961000e+02
5    2018-01-03   AAPL     open  1.692521e+02
6    2018-01-03   AMZN     open  1.188300e+03
7    2018-01-03     FB     open  1.818800e+02
8    2018-01-03   GOOG     open  1.064310e+03
9    2018-01-03   NFLX     open  2.020500e+02
10   2018-01-04   AAPL     open  1.692619e+02
11   2018-01-04   AMZN     open  1.205000e+03
12   2018-01-04     FB     open  1.849000e+02
13   2018-01-04   GOOG     open  1.088000e+03
14   2018-01-04   NFLX     open  2.062000e+02
15   2018-01-05   AAPL     open  1.701448e+02
16   2018-01-05   AMZN     open  1.217510e+03
17   2018-01-05     FB     open  1.855900e+02
18   2018-01-05   GOOG     open  1.094000e+03
19   2018-01-05   NFLX     open  2.072500e+02
20   2018-01-08   AAPL     open  1